## Introduction

This notebook will test the connections to the various containers and packages needed to complete this workshop.  This includes:

- PostgreSQL
- Senzing
- LanceDB
- NetworkX
- Folium

The key part is that the final cell should show ✅ for all connections.  

In [1]:
import importlib
import json
import os
import socket
import time

from IPython.display import IFrame
from IPython.display import HTML

import folium
import grpc
import lancedb
import networkx as nx
import pandas as pd
import psycopg2

from senzing import SzEngineFlags, SzError, SzProduct
from senzing_grpc import SzAbstractFactoryGrpc

## Test PostgreSQL Connection

Connects to the Postgres database using environment variables set in the Docker Compose file and prints the server version.  A failure here usually means the `erkg_postgres` container is not running or is not yet healthy.

In [2]:
try:
    conn = psycopg2.connect(
        host=os.getenv('POSTGRES_HOST', 'postgres'),
        port=os.getenv('POSTGRES_PORT', '5432'),
        user=os.getenv('POSTGRES_USER', 'postgres'),
        password=os.getenv('POSTGRES_PASSWORD', 'workshop'),
        database=os.getenv('POSTGRES_DB', 'erkg')
    )
    cursor = conn.cursor()
    cursor.execute('SELECT version();')
    version = cursor.fetchone()
    print(f"✅ PostgreSQL connection successful!")
    print(f"   Version: {version[0][:50]}...")
    cursor.close()
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL connection failed: {e}")

✅ PostgreSQL connection successful!
   Version: PostgreSQL 15.19 (Debian 15.19-1.pgdg13+2) on x86_...


## Check Senzing Network Reachability

Verifies that the Jupyter container can resolve the `senzing` hostname and open a TCP connection to port 8261.  This is a low-level diagnostic confirming network reachability within the container.  If this fails, the problem is the container infrastructure.

In [3]:
senzing_host = os.getenv('SENZING_GRPC_HOST', 'resolver')
senzing_port = int(os.getenv('SENZING_GRPC_PORT', '8261'))

try:
    # Try to resolve the hostname
    ip_address = socket.gethostbyname(senzing_host)
    print(f"✅ DNS resolution successful: {senzing_host} -> {ip_address}")
    
    # Try to connect to the port
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(5)
    result = sock.connect_ex((senzing_host, senzing_port))
    sock.close()
    
    if result == 0:
        print(f"✅ Resolver service is reachable at {senzing_host}:{senzing_port}")
    else:
        print(f"❌ Resolver service is not responding on port {senzing_port}")
        print(f"   Make sure the resolver container is running:")
        print(f"   Run: docker-compose ps")
        
except socket.gaierror as e:
    print(f"❌ Cannot resolve hostname '{senzing_host}'")
    print(f"   Error: {e}")
    print(f"\n   Troubleshooting steps:")
    print(f"   1. Check all containers are running: docker-compose ps")
    print(f"   2. Check they're on the same network: docker network inspect odsc_west_2026_erkg-network")
    print(f"   3. Restart services: docker-compose restart")
except Exception as e:
    print(f"❌ Connection test failed: {e}")

✅ DNS resolution successful: resolver -> 172.20.0.2
✅ Resolver service is reachable at resolver:8261


## Test Senzing gRPC Connection

This confirms the Senzing application is actually running, initialized, and responding to SDK calls.  Opens a Google Remote Procedure Call (gRPC) the Senzing service and calls `get_version()` to confirm the SDK is reachable and responding.  gRPC (Google Remote Procedure Call) is a framework that lets one program call functions in another program running on a different machine, as if they were local function calls.  It is necessary because Jupyter and Senzing are in different containers, so gRPC bridges this gap between the two.

In [4]:
try:
    # Create gRPC channel with options
    grpc_host = os.getenv('SENZING_GRPC_HOST', 'resolver')
    grpc_port = os.getenv('SENZING_GRPC_PORT', '8261')
    grpc_url = f"{grpc_host}:{grpc_port}"
    
    print(f"   Connecting to: {grpc_url}")
    
    # Create channel with explicit options
    options = [
        ('grpc.enable_http_proxy', 0),
        ('grpc.max_receive_message_length', 100 * 1024 * 1024),  # 100MB
    ]
    channel = grpc.insecure_channel(grpc_url, options=options)
    
    # Wait for channel to be ready (with timeout)
    print("   Waiting for channel to be ready...")
    grpc.channel_ready_future(channel).result(timeout=10)
    
    # Create Senzing factory using the channel
    sz_factory = SzAbstractFactoryGrpc(channel)
    
    # Test SzProduct to get version info
    sz_product = sz_factory.create_product()
    version_info = sz_product.get_version()
    print(f"✅ Senzing gRPC connection successful!")
    print(f"   Connected to: {grpc_url}")
    print(f"   Senzing version info:\n{version_info[:200]}...")
    
except grpc.FutureTimeoutError:
    print(f"❌ Connection timeout - Senzing service not responding")
    print(f"   Check if senzing container is running: docker-compose ps resolver")
except grpc.RpcError as e:
    print(f"❌ gRPC Error: {e.code()}")
    print(f"   Details: {e.details()}")
    if e.code() == grpc.StatusCode.UNAVAILABLE:
        print(f"\n   The resolver service is not available. Please check:")
        print(f"   1. Is the container running? docker-compose ps")
        print(f"   2. Are all services on the same network?")
        print(f"   3. Try restarting: docker-compose restart resolver")
except SzError as e:
    print(f"❌ Senzing SDK error: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {type(e).__name__}: {e}")

   Connecting to: resolver:8261
   Waiting for channel to be ready...
✅ Senzing gRPC connection successful!
   Connected to: resolver:8261
   Senzing version info:
{"PRODUCT_NAME":"Senzing SDK","VERSION":"4.2.0","BUILD_VERSION":"4.2.0.26023","BUILD_DATE":"2026-01-23","BUILD_NUMBER":"2026_01_23__10_29","COMPATIBILITY_VERSION":{"CONFIG_VERSION":"11"},"SCHEMA_VERSI...


## Check Senzing License

Retrieves and displays the Senzing license details, including the expiration date and record limit.  The free evaluation license we are using today allows up to 500 records, which is plenty for this workshop.  A full license was used in order to create the database used for this workshop.

In [5]:
try:
    grpc_url = f"{os.getenv('SENZING_GRPC_HOST', 'resolver')}:{os.getenv('SENZING_GRPC_PORT', '8261')}"
    channel = grpc.insecure_channel(grpc_url)
    sz_factory = SzAbstractFactoryGrpc(channel)
    
    sz_product = sz_factory.create_product()
    license_info = sz_product.get_license()
    print(f"✅ Senzing license verified")
    print(f"   License info:\n{license_info[:200]}...")
    
except SzError as e:
    print(f"❌ License check failed: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

✅ Senzing license verified
   License info:
{"customer":"","contract":"","issueDate":"2026-01-23","licenseType":"EVAL (Solely for non-productive use)","licenseLevel":"","billing":"","expireDate":"2027-01-24","recordLimit":500,"advSearch":0}...


## Confirm database is populated

When you run this cell, you should see that you have the full database needed for this workshop and get the following output:

✅ Workshop data loaded  
   39,413 records resolved into 31,339 entities  
   2,482 geocoded addresses

In [6]:
try:
    grpc_url = f"{os.getenv('SENZING_GRPC_HOST', 'resolver')}:{os.getenv('SENZING_GRPC_PORT', '8261')}"
    channel = grpc.insecure_channel(grpc_url)
    sz_factory = SzAbstractFactoryGrpc(channel)

    license_info = json.loads(sz_factory.create_product().get_license())
    customer = license_info.get('customer') or '(none - free evaluation)'

    print(f"✅ Senzing license verified")
    print(f"   Licensed to:  {customer}")
    print(f"   Record limit: {license_info['recordLimit']:,}")
    print()
    print("   The record limit applies to LOADING records. This workshop only")
    print("   READS an already-resolved database, which is unlimited - so the")
    print("   free 500-record limit is expected and does not affect you.")

    # the check that actually matters: did the database arrive?
    conn = psycopg2.connect(
        host=os.getenv('POSTGRES_HOST', 'postgres'),
        user=os.getenv('POSTGRES_USER', 'postgres'),
        password=os.getenv('POSTGRES_PASSWORD', 'workshop'),
        database=os.getenv('POSTGRES_DB', 'erkg'))
    cur = conn.cursor()
    cur.execute("""SELECT (SELECT count(*) FROM dsrc_record),
                          (SELECT count(*) FROM res_ent),
                          (SELECT count(*) FROM workshop.entity_geo)""")
    records, entities, geocoded = cur.fetchone()
    conn.close()

    print()
    if records == 0:
        print("❌ The database is EMPTY. Run:  docker compose down -v && ./scripts/start.sh")
    else:
        print(f"✅ Workshop data loaded")
        print(f"   {records:,} records resolved into {entities:,} entities")
        print(f"   {geocoded:,} geocoded addresses")

except SzError as e:
    print(f"❌ License check failed: {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")


✅ Senzing license verified
   Licensed to:  (none - free evaluation)
   Record limit: 500

   The record limit applies to LOADING records. This workshop only
   READS an already-resolved database, which is unlimited - so the
   free 500-record limit is expected and does not affect you.

✅ Workshop data loaded
   39,413 records resolved into 31,339 entities
   2,482 geocoded addresses


## What is Actually Inside the Postgres Container

There are **two** "repositories" (a database the engine works against) that we will using with Senzing in this workshop, `sandbox` and `erkg`, and they do different jobs.  There are two different Senzing engines that connect to them, `erkg_sandbox` and `erkg_resolver` respectively.  We will briefly learn about how to use the Senzing SDK in the `sandbox` and then move to the `erkg` for the bulk of this workshop.

| Repository | Engine | Container Name | Data Present |
| --- | --- | --- | --- |
| **ERKG** | `erkg_resolver` | `resolver` | 39,413, already resolved | read only |
| **Sandbox** | `erkg_sandbox` | `sandbox` | empty | add your own |

The main ERKG repository is far past the free 500-record licence limit, so records
cannot be added to it.  The example below demonstrating how to use the SDK happens in the sandbox, where the free
licence is plenty.

In [7]:
sandbox_url = f"{os.getenv('SANDBOX_GRPC_HOST', 'sandbox')}:{os.getenv('SANDBOX_GRPC_PORT', '8261')}"
sandbox_channel = grpc.insecure_channel(sandbox_url)
grpc.channel_ready_future(sandbox_channel).result(timeout=20)

sandbox_factory = SzAbstractFactoryGrpc(sandbox_channel)
sandbox_engine  = sandbox_factory.create_engine()
sandbox_config  = sandbox_factory.create_configmanager()

lic = json.loads(sandbox_factory.create_product().get_license())
print(f"✅ Sandbox connected at {sandbox_url}")
print(f"   Record limit: {lic['recordLimit']:,}  (free licence - expected, and plenty)")

✅ Sandbox connected at sandbox:8261
   Record limit: 500  (free licence - expected, and plenty)


## Configure a Data Source

Every record carries a `DATA_SOURCE` label saying where it came from, and Senzing
rejects records whose label it has not been told about. So before loading
anything, register the name.

**REALLY IMPORTANT!!!**
The Senzing engine only reads its configuration at startup, so after
registering you need to restart the sandbox before it notices:

    docker restart erkg_sandbox

The cell below registers `BURLINGAME-DEMO` and tells you whether a restart is needed.

**If you add this data source for the first time and do not restart after running the following cell, you will get an error!**

In [8]:
DS = 'BURLINGAME-DEMO'

# build from the CURRENT config - never from a template, which would drop
# whatever is already registered
config = sandbox_config.create_config_from_config_id(
    sandbox_config.get_default_config_id())

existing = [d['DSRC_CODE'] for d in
            json.loads(config.get_data_source_registry())['DATA_SOURCES']]
print(f"Already registered: {existing}")

if DS not in existing:
    config.register_data_source(DS)
    new_config_id = sandbox_config.register_config(config.export(), f"Added {DS}")
    sandbox_config.set_default_config_id(new_config_id)
    print(f"✅ Registered {DS} (config {new_config_id})")
else:
    print(f"✅ {DS} already registered")

# Is the running engine using the config we just saved?
default_id = sandbox_config.get_default_config_id()
active_id  = sandbox_engine.get_active_config_id()
print(f"\n   stored config: {default_id}")
print(f"   engine is using: {active_id}")

if default_id != active_id:
    print("\n" + "=" * 62)
    print("The engine is running an OLDER configuration.")
    print("Restart it, then re-run the connection cell above:")
    print("\n    docker restart erkg_sandbox\n")
    print("Without this, loading records fails with:")
    print("    SENZ2207|Data source code [%s] does not exist." % DS)
    print("=" * 62)
else:
    print("\n✅ Engine is up to date - ready to load records")

Already registered: ['TEST', 'SEARCH', 'BURLINGAME-DEMO']
✅ BURLINGAME-DEMO already registered

   stored config: 3006845664
   engine is using: 3006845664

✅ Engine is up to date - ready to load records


## Add Records

To see how to add records to Senzing, we have the following simulated records (that look similar to some of the actual data we will be using for this workshop).  Notice that two of them are the same
business written slightly differently and observe what Senzing does with them.

In [9]:
records = [
    ('1001', [{'RECORD_TYPE': 'ORGANIZATION'},
              {'NAME_ORG': 'BURLINGAME BAGEL COMPANY'},
              {'ADDR_TYPE': 'BUSINESS', 'ADDR_FULL': '100 PRIMROSE RD, BURLINGAME, CA 94010'},
              {'PHONE_TYPE': 'WORK', 'PHONE_NUMBER': '650-555-0100'}]),

    # the same business, written differently - should RESOLVE with 1001
    ('1002', [{'RECORD_TYPE': 'ORGANIZATION'},
              {'NAME_ORG': 'BURLINGAME BAGEL CO'},
              {'ADDR_TYPE': 'BUSINESS', 'ADDR_FULL': '100 PRIMROSE ROAD, BURLINGAME, CA 94010'}]),

    # a different business - should stay separate
    ('1003', [{'RECORD_TYPE': 'ORGANIZATION'},
              {'NAME_ORG': 'PENINSULA COFFEE ROASTERS'},
              {'ADDR_TYPE': 'BUSINESS', 'ADDR_FULL': '250 CALIFORNIA DR, BURLINGAME, CA 94010'}]),

    # a PERSON at the bagel shop's address - same address, different kind of thing
    ('1004', [{'RECORD_TYPE': 'PERSON'},
              {'NAME_FIRST': 'MARIA', 'NAME_LAST': 'GONZALEZ'},
              {'ADDR_TYPE': 'HOME', 'ADDR_FULL': '100 PRIMROSE RD, BURLINGAME, CA 94010'}]),
]

for record_id, features in records:
    record = {'DATA_SOURCE': DS, 'RECORD_ID': record_id, 'FEATURES': features}
    sandbox_engine.add_record(DS, record_id, json.dumps(record))

print(f"✅ Added {len(records)} records to {DS}")

✅ Added 4 records to BURLINGAME-DEMO


## Retrieve Entities

Records go in; **entities** come out. Look up each record and see which entity
it ended up in.

In [10]:
for record_id in ('1001', '1003', '1004'):
    entity = json.loads(
        sandbox_engine.get_entity_by_record_id(DS, record_id))['RESOLVED_ENTITY']
    members = [r['RECORD_ID'] for r in entity['RECORDS']]
    print(f"record {record_id} -> entity {entity['ENTITY_ID']}: {entity['ENTITY_NAME']}")
    print(f"    built from records: {members}\n")

record 1001 -> entity 17: BURLINGAME BAGEL COMPANY
    built from records: ['1001', '1002']

record 1003 -> entity 19: PENINSULA COFFEE ROASTERS
    built from records: ['1003']

record 1004 -> entity 20: MARIA GONZALEZ
    built from records: ['1004']



## Getting Basic ER info

Maria Gonzalez lives at the bagel shop's address. Senzing noticed, but did not
merge them. `why_entities` explains the decision.  Notice that the results indicate a positive match on address (`+ADDRESS`) but a negative match on record type (`-RECORD_TYPE`).  For Senzing, this is enough to say that they are possibly related, but not at a high enough confidence to actually merge them.

In [11]:
bagel  = json.loads(sandbox_engine.get_entity_by_record_id(DS, '1001'))['RESOLVED_ENTITY']['ENTITY_ID']
person = json.loads(sandbox_engine.get_entity_by_record_id(DS, '1004'))['RESOLVED_ENTITY']['ENTITY_ID']

why = json.loads(sandbox_engine.why_entities(
    bagel, person, flags=SzEngineFlags.SZ_WHY_ENTITIES_DEFAULT_FLAGS))
info = why['WHY_RESULTS'][0]['MATCH_INFO']

print(f"Why key:     {info['WHY_KEY']}")
print(f"Match level: {info['MATCH_LEVEL_CODE']}")

Why key:     +ADDRESS-RECORD_TYPE
Match level: POSSIBLY_RELATED


## Search by Attributes

You do not need a record ID to find something. Search on whatever you know.

In [12]:
results = json.loads(sandbox_engine.search_by_attributes(
    json.dumps({'NAME_ORG': 'BURLINGAME BAGEL'})))

for hit in results.get('RESOLVED_ENTITIES', []):
    entity = hit['ENTITY']['RESOLVED_ENTITY']
    print(f"{entity['ENTITY_NAME']}  (entity {entity['ENTITY_ID']})")
    print(f"    match key: {hit['MATCH_INFO']['MATCH_KEY']}")

BURLINGAME BAGEL COMPANY  (entity 17)
    match key: +NAME


## Clean Up

Leave the sandbox empty so this notebook can be re-run from scratch.
(Re-running the load cell without this is safe too — `add_record` replaces by
record ID.)

In [13]:
for record_id in ('1001', '1002', '1003', '1004'):
    sandbox_engine.delete_record(DS, record_id)
print("✅ Sandbox reset")

✅ Sandbox reset


## The Main Repository

Now we are going to connect to the main repository, which has the geo-ERKG databse that we will use for the rest of the workshop. 

In [14]:
# The MAIN repository: 39,413 records, already entity-resolved and geocoded.
# Read-only for you - it is far past the free 500-record licence limit, so
# records cannot be added here. That is what the sandbox is for.

main_url = f"{os.getenv('SENZING_GRPC_HOST', 'resolver')}:{os.getenv('SENZING_GRPC_PORT', '8261')}"
main_channel = grpc.insecure_channel(main_url)
grpc.channel_ready_future(main_channel).result(timeout=20)

sz_factory = SzAbstractFactoryGrpc(main_channel)
sz_engine = sz_factory.create_engine()
sz_configmanager = sz_factory.create_configmanager()

print(f"✅ Main repository connected at {main_url}")

✅ Main repository connected at resolver:8261


## Information on the ERKG repository

The following gives you some basic information on what is contained in that repository, shown as the number of records in each data source in the PostgreSQL.

In [15]:
config = sz_configmanager.create_config_from_config_id(
    sz_configmanager.get_default_config_id())
sources = {d['DSRC_ID']: d['DSRC_CODE']
           for d in json.loads(config.get_data_source_registry())['DATA_SOURCES']}

conn = psycopg2.connect(
    host=os.getenv('POSTGRES_HOST', 'postgres'),
    user=os.getenv('POSTGRES_USER', 'postgres'),
    password=os.getenv('POSTGRES_PASSWORD', 'workshop'),
    database=os.getenv('POSTGRES_DB', 'erkg'))
cur = conn.cursor()

cur.execute("SELECT dsrc_id, count(*) FROM dsrc_record GROUP BY 1 ORDER BY 2 DESC")
counts = cur.fetchall()

print(f"   {'data source':26}{'records':>10}")
print("   " + "-" * 36)
for dsrc_id, n in counts:
    print(f"   {sources.get(dsrc_id, dsrc_id):26}{n:>10,}")

cur.execute("SELECT count(*) FROM dsrc_record");  total    = cur.fetchone()[0]
cur.execute("SELECT count(*) FROM res_ent");      entities = cur.fetchone()[0]
cur.execute("SELECT count(*), count(DISTINCT entity_id) FROM workshop.entity_geo")
geo_rows, geo_entities = cur.fetchone()
conn.close()

print("   " + "-" * 36)
print(f"   {'TOTAL':26}{total:>10,}")
print(f"\n   resolved into {entities:,} entities  (compression {total/entities:.2f})")
print(f"   {geo_rows:,} geocoded addresses on {geo_entities:,} entities")

# a concrete proof that resolution actually happened across sources
entity = json.loads(
    sz_engine.get_entity_by_record_id('BURLINGAME', '20508711'))['RESOLVED_ENTITY']
srcs = sorted({r['DATA_SOURCE'] for r in entity['RECORDS']})
print(f"\n   spot check -> entity {entity['ENTITY_ID']}: {entity['ENTITY_NAME']}")
print(f"   {len(entity['RECORDS'])} records from {srcs}")


   data source                  records
   ------------------------------------
   CA-CONTRACTOR-PERSONNEL       10,215
   CA-SOS-FILINGS                 7,801
   CA-SOS-AGENTS                  7,421
   CA-SOS-PRINCIPALS              5,701
   BURLINGAME                     4,781
   CA-CONTRACTORS                 3,274
   ABC-RETAIL                       201
   ABC-NONRETAIL                     19
   ------------------------------------
   TOTAL                         39,413

   resolved into 31,339 entities  (compression 1.26)
   2,482 geocoded addresses on 2,401 entities

   spot check -> entity 4372: TOPGOLF
   3 records from ['ABC-RETAIL', 'BURLINGAME']


## Test LanceDB

We will use LanceDB for our vector database for the non-ERKG RAG notebook.  This cell creates a small test table in LanceDB, runs a vector similarity query, then drops the table.  This confirms that the vector store is mounted and working before we use it to store entity embeddings later in the workshop.

In [16]:
try:
    # Create a test database
    db_path = "/workspace/lancedb_data/test_db"
    db = lancedb.connect(db_path)
    
    # Create a simple test table
    test_data = pd.DataFrame({
        "id": [1, 2, 3],
        "text": ["hello", "world", "test"],
        "vector": [[0.1, 0.2], [0.3, 0.4], [0.5, 0.6]]
    })
    
    table = db.create_table("test_table", test_data, mode="overwrite")
    print(f"✅ LanceDB initialized successfully!")
    print(f"   Database path: {db_path}")
    print(f"   Created test table with {len(table)} rows")
    
    # Verify we can query
    results = table.search([0.2, 0.3]).limit(2).to_pandas()
    print(f"   Query test: Retrieved {len(results)} rows")
    
    # Clean up
    db.drop_table("test_table")
    print("✅ Test table cleaned up")
    
except Exception as e:
    print(f"❌ LanceDB test failed: {e}")

✅ LanceDB initialized successfully!
   Database path: /workspace/lancedb_data/test_db
   Created test table with 3 rows
   Query test: Retrieved 2 rows
✅ Test table cleaned up


[2026-10-12T20:01:26Z WARN  lance::dataset::write::insert] No existing dataset at /workspace/lancedb_data/test_db/test_table.lance, it will be created


## Test NetworkX

We will be creating our knowledge graphs in memory using NetworkX.  This cell builds a small four-node graph and computes basic metrics like degree and shortest path.  This is a quick check that the NetworkX package is installed and functional.

In [17]:
try:
    # Create a simple test graph
    G = nx.Graph()
    G.add_edges_from([(1, 2), (2, 3), (3, 4), (4, 1)])
    
    # Calculate some basic metrics
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    avg_degree = sum(dict(G.degree()).values()) / num_nodes
    
    print(f"✅ NetworkX working!")
    print(f"   Created test graph:")
    print(f"   - Nodes: {num_nodes}")
    print(f"   - Edges: {num_edges}")
    print(f"   - Average degree: {avg_degree:.2f}")
    
    # Test shortest path
    path = nx.shortest_path(G, 1, 3)
    print(f"   - Shortest path 1→3: {path}")
    
except Exception as e:
    print(f"❌ NetworkX test failed: {e}")

✅ NetworkX working!
   Created test graph:
   - Nodes: 4
   - Edges: 4
   - Average degree: 2.00
   - Shortest path 1→3: [1, 2, 3]


## Test Folium

Notebook 03 draws the whole knowledge graph on a map using Folium, so this cell
renders a miniature version of it: two businesses as circles and a line between
them, centred on the conference hotel.

Make sure you can see the **map tiles** load, not just the circles, and that you
can pan and zoom. If the circles appear but the map behind them is blank, the
browser cannot reach the tile server — check your network rather than the
container.

In [18]:
try:
    # Same ingredients as notebook 03: a tile layer, circle markers positioned
    # from coordinates, and a line joining two of them.
    HYATT = (37.59086, -122.35975)          # 1333 Bayshore Hwy, the conference hotel
    OTHER = (37.59184, -122.36949)          # 1003 Edwards Rd, about 900 m away

    m = folium.Map(location=HYATT, zoom_start=15, tiles="OpenStreetMap")

    folium.CircleMarker(HYATT, radius=10, color="#c2185b", fill=True,
                        fill_opacity=0.9,
                        tooltip="Hyatt Regency SFO").add_to(m)
    folium.CircleMarker(OTHER, radius=7, color="#0b0b0b", fill=True,
                        fill_opacity=0.9,
                        tooltip="another business").add_to(m)
    folium.PolyLine([HYATT, OTHER], color="#4a3aa7", weight=2,
                    dash_array="8,6").add_to(m)

    html = m._repr_html_()
    assert "leaflet" in html.lower(), "Leaflet JavaScript missing from the map"

    print("✅ Folium working!")
    print(f"   Rendered a map with 2 markers and 1 connecting line")
    print(f"   Interactive map below - pan and zoom to confirm the tiles load\n")

    display(m)

except Exception as e:
    print(f"❌ Folium test failed: {e}")

✅ Folium working!
   Rendered a map with 2 markers and 1 connecting line
   Interactive map below - pan and zoom to confirm the tiles load



## Verify All Workshop Packages

Iterates through every required Python package and reports whether it is installed.  Run this last to quickly see what is missing.

In [19]:
print("Checking all workshop Python packages...")
print()

packages = {
    'senzing_grpc': 'Senzing gRPC Client',
    'senzing': 'Senzing SDK',
    'psycopg2': 'PostgreSQL Driver',
    'sqlalchemy': 'SQLAlchemy',
    'lancedb': 'LanceDB Vector Store',
    'fastembed': 'FastEmbed Embeddings',
    'pandas': 'Pandas Data Analysis',
    'networkx': 'NetworkX Graph Analysis',
    'folium': 'Folium Map Visualization',
    'placekey': 'Placekey Geocoding',
    'anthropic': 'Anthropic Claude Client',
    'openai': 'OpenAI Client',
    'dotenv': 'Python DotEnv'
}

all_good = True
for pkg, description in packages.items():
    try:
        importlib.import_module(pkg)
        print(f"✅ {description:30s} ({pkg})")
    except ImportError:
        print(f"❌ {description:30s} ({pkg}) - NOT INSTALLED")
        all_good = False

print()
if all_good:
    print("✅ All required packages installed!")
else:
    print("❌ Some packages are missing. Check Dockerfile and rebuild.")

Checking all workshop Python packages...

✅ Senzing gRPC Client            (senzing_grpc)
✅ Senzing SDK                    (senzing)
✅ PostgreSQL Driver              (psycopg2)
✅ SQLAlchemy                     (sqlalchemy)
✅ LanceDB Vector Store           (lancedb)
✅ FastEmbed Embeddings           (fastembed)
✅ Pandas Data Analysis           (pandas)
✅ NetworkX Graph Analysis        (networkx)
✅ Folium Map Visualization       (folium)
✅ Placekey Geocoding             (placekey)
✅ Anthropic Claude Client        (anthropic)
✅ OpenAI Client                  (openai)
✅ Python DotEnv                  (dotenv)

✅ All required packages installed!
